## Introduktion 

Denne lektion vil dække: 
- Hvad funktionkald er, og dets anvendelsestilfælde 
- Hvordan man opretter et funktionkald ved hjælp af Azure OpenAI 
- Hvordan man integrerer et funktionkald i en applikation 

## Læringsmål 

Efter at have gennemført denne lektion vil du vide, hvordan man og forstå: 

- Formålet med at bruge funktionkald 
- Opsæt funktionkald ved hjælp af Azure Open AI-tjenesten 
- Design effektive funktionkald til dit applikations brugstilfælde 


## Forståelse af Funktionskald

Til denne lektion vil vi bygge en funktion til vores uddannelsesstartup, som giver brugere mulighed for at bruge en chatbot til at finde tekniske kurser. Vi vil anbefale kurser, der passer til deres færdighedsniveau, nuværende rolle og teknologiinteresse.

For at gennemføre dette vil vi bruge en kombination af:
 - `Azure Open AI` til at skabe en chatoplevelse for brugeren
 - `Microsoft Learn Catalog API` til at hjælpe brugere med at finde kurser baseret på brugerens forespørgsel
 - `Function Calling` til at tage brugerens forespørgsel og sende den til en funktion for at lave API-anmodningen.

For at komme i gang, lad os se på, hvorfor vi overhovedet ville bruge funktionskald:


### Hvorfor Funktionsopkald 

Hvis du har gennemført en anden lektion i dette kursus, forstår du sandsynligvis kraften i at bruge Store Sprogmodeller (LLMs). Forhåbentlig kan du også se nogle af deres begrænsninger. 

Funktionsopkald er en funktion i Azure Open AI Service for at overvinde følgende begrænsninger: 
1) Konsistent svarformat 
2) Evnen til at bruge data fra andre kilder i en applikation i en chat-kontekst 

Før funktionsopkald var svarene fra en LLM ustrukturerede og inkonsistente. Udviklere var nødt til at skrive kompleks valideringskode for at sikre, at de kunne håndtere hver variation af et svar. 

Brugere kunne ikke få svar som "Hvordan er vejret i Stockholm lige nu?". Dette skyldes, at modeller var begrænset til det tidspunkt, hvor dataene blev trænet. 

Lad os se på eksemplet nedenfor, der illustrerer dette problem: 

Lad os sige, at vi ønsker at oprette en database over studerendes data, så vi kan foreslå det rigtige kursus til dem. Nedenfor har vi to beskrivelser af studerende, der er meget ens i de data, de indeholder.


In [ ]:
student_1_description="Emily Johnson is a sophomore majoring in computer science at Duke University. She has a 3.7 GPA. Emily is an active member of the university's Chess Club and Debate Team. She hopes to pursue a career in software engineering after graduating."
 
student_2_description = "Michael Lee is a sophomore majoring in computer science at Stanford University. He has a 3.8 GPA. Michael is known for his programming skills and is an active member of the university's Robotics Club. He hopes to pursue a career in artificial intelligence after finishing his studies."


Vi vil sende dette til en LLM for at analysere dataene. Dette kan senere bruges i vores applikation til at sende det til en API eller gemme det i en database. 

Lad os lave to identiske prompts, hvor vi instruerer LLM om, hvilken information vi er interesserede i: 


Vi vil sende dette til en LLM for at analysere de dele, der er vigtige for vores produkt. Så vi kan oprette to identiske prompts til at instruere LLM: 


In [ ]:
prompt1 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_1_description}
'''


prompt2 = f'''
Please extract the following information from the given text and return it as a JSON object:

name
major
school
grades
club

This is the body of text to extract the information from:
{student_2_description}
'''


Efter at have oprettet disse to prompts, sender vi dem til LLM ved at bruge `client.responses.create`. Vi gemmer prompten i variablen `input` og tildeler rollen til `user`. Dette er for at efterligne en besked fra en bruger, der bliver skrevet til en chatbot. 



In [ ]:
import os
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

# The OpenAI client points at the Azure OpenAI (Microsoft Foundry) /openai/v1/ endpoint
client = OpenAI(
  api_key=os.environ['AZURE_OPENAI_API_KEY'],
  base_url=f"{os.environ['AZURE_OPENAI_ENDPOINT'].rstrip('/')}/openai/v1/",
  )

deployment=os.environ['AZURE_OPENAI_DEPLOYMENT']


: 

Nu kan vi sende begge forespørgsler til LLM'en og undersøge det svar, vi modtager. 


In [ ]:
openai_response1 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt1}],
 store=False,
)
openai_response1.output_text 


In [ ]:
openai_response2 = client.responses.create(
 model=deployment,    
 input = [{'role': 'user', 'content': prompt2}],
 store=False,
)
openai_response2.output_text


In [ ]:
# Loading the response as a JSON object
json_response1 = json.loads(openai_response1.output_text)
json_response1


In [ ]:
# Loading the response as a JSON object
json_response2 = json.loads(openai_response2.output_text)
json_response2



Selvom promptsene er de samme og beskrivelserne ligner hinanden, kan vi få forskellige formater af egenskaben `Grades`. 

Hvis du kører ovenstående celle flere gange, kan formatet være `3.7` eller `3.7 GPA`. 

Dette skyldes, at LLM tager ustrukturerede data i form af den skrevne prompt og også returnerer ustrukturerede data. Vi har brug for et struktureret format, så vi ved, hvad vi kan forvente, når vi gemmer eller bruger disse data.

Ved at bruge funktionel kald kan vi sikre, at vi modtager strukturerede data tilbage. Når vi bruger funktionskald, kalder eller kører LLM faktisk ikke nogen funktioner. I stedet opretter vi en struktur, som LLM skal følge for sine svar. Vi bruger derefter disse strukturerede svar til at vide, hvilken funktion vi skal køre i vores applikationer.  
 


![Funktionskald Flowdiagram](../../../../translated_images/da/Function-Flow.083875364af4f4bb.webp)


Vi kan så tage det, der returneres fra funktionen, og sende det tilbage til LLM. LLM vil derefter svare med naturligt sprog for at besvare brugerens forespørgsel. 


### Brugstilfælde for brug af funktionskald 

**Kald af eksterne værktøjer** 
Chatbots er gode til at give svar på spørgsmål fra brugere. Ved at bruge funktionskald kan chatbots bruge beskeder fra brugere til at udføre bestemte opgaver. For eksempel kan en studerende bede chatbotten om at "Sende en e-mail til min underviser og sige, at jeg har brug for mere hjælp med dette emne". Dette kan lave et funktionskald til `send_email(to: string, body: string)`


**Opret API- eller databaseforespørgsler**
Brugere kan finde information ved hjælp af naturligt sprog, som konverteres til en formateret forespørgsel eller API-anmodning. Et eksempel på dette kunne være en lærer, der spørger "Hvem er de studerende, der har gennemført den sidste opgave", hvilket kunne kalde en funktion kaldet `get_completed(student_name: string, assignment: int, current_status: string)`


**Oprettelse af strukturerede data**
Brugere kan tage et tekstafsnit eller CSV og bruge LLM til at udtrække vigtig information fra det. For eksempel kan en studerende konvertere en Wikipedia-artikel om fredsaftaler til at lave AI-flashkort. Dette kan gøres ved at bruge en funktion kaldet `get_important_facts(agreement_name: string, date_signed: string, parties_involved: list)`


## 2. Oprettelse af dit første funktionskald 

Processen med at oprette et funktionskald inkluderer 3 hovedtrin: 
1. Kald Chat Completions API'en med en liste over dine funktioner og en brugermeddelelse 
2. Læs modellens svar for at udføre en handling, dvs. udfør en funktion eller API-kald 
3. Lav et nyt kald til Chat Completions API'en med svaret fra din funktion for at bruge den information til at skabe et svar til brugeren. 


![Flowet af et Funktionskald](../../../../translated_images/da/LLM-Flow.3285ed8caf4796d7.webp)


### Elementer i et funktionskald 

#### Brugernes Input 

Det første skridt er at oprette en brugermeddelelse. Denne kan dynamisk tildeles ved at tage værdien af en tekstinput, eller du kan tildele en værdi her. Hvis dette er første gang, du arbejder med Chat Completions API'en, skal vi definere `role` og `content` i meddelelsen. 

`role` kan enten være `system` (opretter regler), `assistant` (modellen) eller `user` (slutbrugeren). Ved funktionskald tildeler vi dette som `user` og et eksempelspørgsmål. 


In [ ]:
messages= [ {"role": "user", "content": "Find me a good course for a beginner student to learn Azure."} ]

### Oprettelse af funktioner. 

Næste skridt er at definere en funktion og parametrene for den funktion. Vi vil kun bruge én funktion her kaldet `search_courses`, men du kan oprette flere funktioner.

**Vigtigt**: Funktioner er inkluderet i systembeskeden til LLM og tæller med i antallet af tilgængelige tokens, du har til rådighed. 


In [ ]:
# The Responses API uses a flat tool format: name/description/parameters at the top level
functions = [
   {
      "type":"function",
      "name":"search_courses",
      "description":"Retrieves courses from the search index based on the parameters provided",
      "parameters":{
         "type":"object",
         "properties":{
            "role":{
               "type":"string",
               "description":"The role of the learner (i.e. developer, data scientist, student, etc.)"
            },
            "product":{
               "type":"string",
               "description":"The product that the lesson is covering (i.e. Azure, Power BI, etc.)"
            },
            "level":{
               "type":"string",
               "description":"The level of experience the learner has prior to taking the course (i.e. beginner, intermediate, advanced)"
            }
         },
         "required":[
            "role"
         ]
      }
   }
]


**Definitioner** 

`name` - Navnet på funktionen, som vi ønsker skal kaldes. 

`description` - Dette er beskrivelsen af, hvordan funktionen fungerer. Her er det vigtigt at være specifik og klar 

`parameters` - En liste over værdier og format, som du ønsker, at modellen skal producere i sit svar 


`type` - Den datatype, som egenskaberne vil blive gemt i 

`properties` - Liste over de specifikke værdier, som modellen vil bruge til sit svar 


`name` - navnet på egenskaben, som modellen vil bruge i sit formaterede svar 

`type` - Datatypen for denne egenskab 

`description` - Beskrivelse af den specifikke egenskab 


**Valgfrit**

`required` - påkrævet egenskab for, at funktionskaldet kan fuldføres 


### Foretage funktionsopkaldet 
Efter at have defineret en funktion skal vi nu inkludere den i kaldet til Chat Completion API'en. Det gør vi ved at tilføje `functions` til forespørgslen. I dette tilfælde `functions=functions`. 

Der er også en mulighed for at sætte `function_call` til `auto`. Det betyder, at vi lader LLM beslutte, hvilken funktion der skal kaldes baseret på brugermeldingen i stedet for at tildele det selv.


In [ ]:
response = client.responses.create(model=deployment, 
                                        input=messages,
                                        tools=functions, 
                                        tool_choice="auto",
                                        store=False) 

print(response.output)


Lad os nu se på svaret og se, hvordan det er formateret: 

{
  "role": "assistant",
  "function_call": {
    "name": "search_courses",
    "arguments": "{\n  \"role\": \"student\",\n  \"product\": \"Azure\",\n  \"level\": \"beginner\"\n}"
  }
}

Du kan se, at funktionen bliver kaldt ved navn, og ud fra brugerbeskeden var LLM i stand til at finde dataene, der passer til funktionens argumenter. 


## 3. Integrering af funktionskald i en applikation. 


Efter vi har testet det formaterede svar fra LLM, kan vi nu integrere dette i en applikation. 

### Styring af flowet 

For at integrere dette i vores applikation, lad os tage følgende trin: 

Først, lad os lave opkaldet til Open AI-tjenesterne og gemme beskeden i en variabel kaldet `response_message`. 


In [ ]:
# Extract the function call items from the response output
tool_calls = [item for item in response.output if item.type == "function_call"]


Nu vil vi definere funktionen, som vil kalde Microsoft Learn API'et for at få en liste over kurser: 


In [ ]:
import requests

def search_courses(role, product, level):
    url = "https://learn.microsoft.com/api/catalog/"
    params = {
        "role": role,
        "product": product,
        "level": level
    }
    response = requests.get(url, params=params)
    modules = response.json()["modules"]
    results = []
    for module in modules[:5]:
        title = module["title"]
        url = module["url"]
        results.append({"title": title, "url": url})
    return str(results)



Som en bedste praksis vil vi derefter se, om modellen ønsker at kalde en funktion. Derefter opretter vi en af de tilgængelige funktioner og matcher den til den funktion, der bliver kaldt.
Vi tager derefter argumenterne til funktionen og kortlægger dem til argumenter fra LLM.

Endelig tilføjer vi funktionskaldsmeddelelsen og de værdier, der blev returneret af `search_courses`-meddelelsen. Dette giver LLM alle de oplysninger, den har brug for til at
svare brugeren ved hjælp af naturligt sprog.


In [ ]:
# Check if the model wants to call a function
if tool_calls:
    tool_call = tool_calls[0]
    print("Recommended Function call:")
    print(tool_call.name)
    print()

    # Call the function. 
    function_name = tool_call.name

    available_functions = {
            "search_courses": search_courses,
    }
    function_to_call = available_functions[function_name] 

    function_args = json.loads(tool_call.arguments)
    function_response = function_to_call(**function_args)

    print("Output of function call:")
    print(function_response)
    print(type(function_response))


    # Add the model's function call item(s) and our function result to the conversation.
    # The Responses API represents tool results as `function_call_output` items.
    messages.extend(response.output)  # adding the model's function_call item(s)
    messages.append( # adding function response to messages
        {
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": function_response,
        }
    )


Nu sender vi den opdaterede besked til LLM, så vi kan modtage et svar på naturligt sprog i stedet for et API JSON-formateret svar. 


In [ ]:
print("Messages in next request:")
print(messages)
print()

second_response = client.responses.create(
    input=messages,
    model=deployment,
    tools=functions,
    tool_choice="auto",
    temperature=0,
    store=False,
        )  # get a new response from the model where it can see the function response


print(second_response.output_text)


## Kodeudfordring 

Godt klaret! For at fortsætte din læring om Azure Open AI Function Calling kan du bygge: https://learn.microsoft.com/training/support/catalog-api-developer-reference?WT.mc_id=academic-105485-koreyst 
 - Flere parametre til funktionen, der kan hjælpe elever med at finde flere kurser. Du kan finde de tilgængelige API-parametre her: 
 - Opret et andet funktionskald, der tager mere information fra eleven som deres modersmål 
 - Opret fejlbehandling, når funktionskaldet og/eller API-kaldet ikke returnerer passende kurser 


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
